# Pitch Quality — Baseline (Phase 3: EDA → cleaning → features → baseline models)

**Question:** how much run value does a pitch carry from its physics alone (velo, movement, spin, release), before the result is known?

- **Data:** Statcast regular season 2020–2026, slim columns, one parquet per month in the public `statcast` Supabase bucket (built by `python -m soccerhub.pipelines.statcast --upload`).
- **Already done upstream:** deduped on `(game_pk, at_bat_number, pitch_number)`, `game_type == 'R'` only, **left-handers mirrored** (`pfx_x`, `release_pos_x`, `vx0`, `ax` negated, `spin_axis → 360 − axis`), so every pitcher reads as a righty. `plate_x` and `arm_angle` are raw.
- **Target:** `rv = −delta_run_exp` (Statcast's `delta_run_exp` is the batter's view; + here = good for the pitcher).
- **Split:** train 2020–24 · validate 2025 · test 2026, plus a GroupKFold-by-pitcher check.
- **The real test:** does 2025 predicted quality forecast 2026 results better than 2025 results do?

In [ ]:
# Colab: xgboost ships preinstalled; shap may not
!pip -q install shap

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb

SUPABASE_URL = "https://rgnzemkdldohplwbxxwy.supabase.co"  # public project URL (same one the site uses)
BASE = f"{SUPABASE_URL}/storage/v1/object/public/statcast/pitches"
SEASONS = range(2020, 2027)
TRAIN, VAL, TEST = range(2020, 2025), [2025], [2026]

## 1. Load

In [ ]:
parts = []
for season in SEASONS:
    for month in range(3, 12):
        try:
            parts.append(pd.read_parquet(f"{BASE}/{season}/{month:02d}.parquet").assign(season=season))
        except Exception:  # month with no regular-season games (e.g. 2020 before July) -> no file
            pass
raw = pd.concat(parts, ignore_index=True)
raw["game_date"] = pd.to_datetime(raw.game_date)
print(f"{len(raw):,} pitches, {raw.memory_usage(deep=True).sum() / 1e9:.2f} GB in memory")
raw.groupby("season").size().rename("pitches").to_frame().T

## 2. EDA

In [ ]:
# Pitch mix by season (%)
mix = pd.crosstab(raw.season, raw.pitch_type.fillna("null"), normalize="index").mul(100).round(1)
mix[mix.mean().sort_values(ascending=False).index]

In [ ]:
# Null rate of every modelling column, by season — arm_angle coverage decides whether 2020-21 stay
PHYS = ["release_speed", "release_spin_rate", "spin_axis", "pfx_x", "pfx_z", "release_pos_x",
        "release_pos_z", "release_extension", "arm_angle", "delta_run_exp"]
raw.groupby("season")[PHYS].apply(lambda d: d.isna().mean().mul(100)).round(2)

In [ ]:
# Physics by pitch type (inches of break; hb + = arm side after the LHP mirror)
view = raw.assign(ivb_in=raw.pfx_z * 12, hb_arm_in=-raw.pfx_x * 12)
main_types = raw.pitch_type.value_counts().index[:10]
view[view.pitch_type.isin(main_types)].groupby("pitch_type")[
    ["release_speed", "release_spin_rate", "ivb_in", "hb_arm_in", "spin_axis", "arm_angle"]].median().round(1)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for t in ["FF", "SI", "FC", "SL", "ST", "CU", "CH", "FS"]:
    d = view[view.pitch_type == t].sample(min(4000, (view.pitch_type == t).sum()), random_state=0)
    ax[0].scatter(d.hb_arm_in, d.ivb_in, s=2, alpha=.3, label=t)
ax[0].set(xlabel="horizontal break, in (+ arm side)", ylabel="induced vertical break, in", title="Movement by pitch type")
ax[0].legend(markerscale=6, fontsize=8)
s = view.sample(200_000, random_state=0)
ax[1].scatter(s.arm_angle, s.hb_arm_in, s=1, alpha=.05)
ax[1].set(xlabel="arm angle (deg)", ylabel="hb, in", title="Arm angle vs horizontal break")
ax[2].hist(-raw.delta_run_exp.dropna(), bins=200, range=(-1.5, 0.6))
ax[2].set(yscale="log", xlabel="rv = -delta_run_exp", title="Target: spike near 0, long tails")
plt.tight_layout()

In [ ]:
# Target is mostly noise per pitch — this is why we judge the model on aggregates, not per-pitch R²
rv = -raw.delta_run_exp
print(rv.describe().round(4))
print("share of |rv| < 0.1:", round((rv.abs() < 0.1).mean(), 3))

## 3. Cleaning

| Rule | Why |
|---|---|
| Drop null target (`automatic_ball/strike`, pitch-clock violations) | No pitch was thrown |
| Drop null core physics | Model can't score the pitch |
| Drop `pitch_type` ∈ PO, IN, EP, FA, UN, AB, CS, null | Pickoffs, intentional balls, eephus / position-player lobs, unknown |
| Drop position players pitching | Pitcher-seasons with EP+FA share > 50%, or < 100 pitches with fastball velo < 85 |
| Merge rare types SV→CU, FO→FS, SC→CH; drop anything else < 0.5% share | Too few pitches to learn a type |

In [ ]:
df = raw.copy()
n0 = len(df)
log = {}

df = df[df.delta_run_exp.notna()]; log["null target"] = n0 - len(df); n = len(df)
CORE = ["release_speed", "release_spin_rate", "spin_axis", "pfx_x", "pfx_z", "release_pos_x", "release_pos_z", "release_extension"]
df = df.dropna(subset=CORE); log["null core physics"] = n - len(df); n = len(df)

# position players: computed on the pitch types before junk types are dropped
ps = raw.groupby(["season", "pitcher"]).agg(
    pitches=("pitch_type", "size"),
    junk=("pitch_type", lambda s: s.isin(["EP", "FA"]).mean()),
    fb_velo=("release_speed", lambda s: s[raw.loc[s.index, "pitch_type"].isin(["FF", "SI"])].mean()))
position = ps[(ps.junk > 0.5) | ((ps.pitches < 100) & (ps.fb_velo.fillna(0) < 85))].index
df = df[~df.set_index(["season", "pitcher"]).index.isin(position)]; log["position players"] = n - len(df); n = len(df)

df = df[~df.pitch_type.isin(["PO", "IN", "EP", "FA", "UN", "AB", "CS"]) & df.pitch_type.notna()]
log["junk pitch types"] = n - len(df); n = len(df)

df["pitch_type"] = df.pitch_type.replace({"SV": "CU", "FO": "FS", "SC": "CH"})
share = df.pitch_type.value_counts(normalize=True)
df = df[df.pitch_type.isin(share[share >= 0.005].index)]; log["rare types"] = n - len(df)

print(pd.Series(log).to_string(), f"\nkept {len(df):,} / {n0:,} ({len(df) / n0:.1%})")
print(sorted(df.pitch_type.unique()))

## 4. Features

- **Raw physics:** velo, spin, spin axis (as sin/cos — it's an angle), IVB, HB, release x/z, extension, arm angle.
- **Fastball differentials:** each pitch vs the pitcher's primary fastball (most-used FF/SI that season): Δvelo, ΔIVB, ΔHB. A changeup is judged by how far it sits from *this* pitcher's heater.
- **Context:** `pitch_type` one-hot, same-hand flag (`p_throws == stand`).
- **Left out:** count (balls/strikes) and location. This is a *stuff* model — physics only. Location and count belong to a later Pitching+ variant.

In [ ]:
df["ivb_in"], df["hb_arm_in"] = df.pfx_z * 12, -df.pfx_x * 12
rad = np.deg2rad(df.spin_axis)
df["axis_sin"], df["axis_cos"] = np.sin(rad), np.cos(rad)
df["same_hand"] = (df.p_throws == df.stand).astype(int)

fb = df[df.pitch_type.isin(["FF", "SI"])]
primary = (fb.groupby(["season", "pitcher", "pitch_type"]).size().rename("n").reset_index()
             .sort_values("n").drop_duplicates(["season", "pitcher"], keep="last"))
fb_stats = (fb.merge(primary[["season", "pitcher", "pitch_type"]])
              .groupby(["season", "pitcher"])[["release_speed", "ivb_in", "hb_arm_in"]].mean()
              .add_prefix("fb_").reset_index())
df = df.merge(fb_stats, on=["season", "pitcher"], how="left")
df["d_velo"] = df.release_speed - df.fb_release_speed
df["d_ivb"] = df.ivb_in - df.fb_ivb_in
df["d_hb"] = df.hb_arm_in - df.fb_hb_arm_in
print("no primary fastball:", f"{df.fb_release_speed.isna().mean():.2%} of pitches")

NUM = ["release_speed", "release_spin_rate", "axis_sin", "axis_cos", "ivb_in", "hb_arm_in",
       "release_pos_x", "release_pos_z", "release_extension", "arm_angle", "d_velo", "d_ivb", "d_hb", "same_hand"]
X = pd.concat([df[NUM].astype("float32"), pd.get_dummies(df.pitch_type, prefix="pt", dtype="float32")], axis=1)
y = (-df.delta_run_exp).astype("float32")
FEATS = list(X.columns)

tr, va, te = df.season.isin(TRAIN), df.season.isin(VAL), df.season.isin(TEST)
print({k: f"{m.sum():,}" for k, m in [("train", tr), ("val", va), ("test", te)]})

## 5. Models: type-mean baseline → Ridge → XGBoost

In [ ]:
def score(name, pred, mask):
    return {"model": name, "split": "val" if mask is va else "test",
            "rmse": float(np.sqrt(mean_squared_error(y[mask], pred))),
            "r2": r2_score(y[mask], pred)}

results, preds = [], {}

type_mean = y[tr].groupby(df.pitch_type[tr]).mean()
for m in (va, te):
    p = df.pitch_type[m].map(type_mean).fillna(y[tr].mean()).values
    results.append(score("type mean", p, m))

ridge = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=10.0)).fit(X[tr], y[tr])
for m in (va, te):
    results.append(score("ridge", ridge.predict(X[m]), m))

booster = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.03, max_depth=6, min_child_weight=200,
                           subsample=0.8, colsample_bytree=0.8, tree_method="hist", early_stopping_rounds=100,
                           eval_metric="rmse")
booster.fit(X[tr], y[tr], eval_set=[(X[va], y[va])], verbose=200)
for m in (va, te):
    results.append(score("xgboost", booster.predict(X[m]), m))

df["pred"] = booster.predict(X)
pd.DataFrame(results).pivot(index="model", columns="split", values=["rmse", "r2"]).round(5)

Per-pitch R² will be tiny (~0.5–2%): one pitch's outcome is mostly luck. The useful question is whether the
*average* predicted quality of a pitcher's pitch is a stable, forward-looking signal. That's next.

In [ ]:
# Same features, grouped by pitcher: no pitcher appears in both train and test folds (2024 only, for speed)
sub = df.season == 2024
gkf_r2 = []
for f_tr, f_te in GroupKFold(n_splits=5).split(X[sub], y[sub], groups=df.pitcher[sub]):
    m = xgb.XGBRegressor(n_estimators=booster.best_iteration + 1, learning_rate=0.03, max_depth=6,
                         min_child_weight=200, subsample=0.8, colsample_bytree=0.8, tree_method="hist")
    m.fit(X[sub].iloc[f_tr], y[sub].iloc[f_tr])
    gkf_r2.append(r2_score(y[sub].iloc[f_te], m.predict(X[sub].iloc[f_te])))
print("GroupKFold-by-pitcher R² (2024):", np.round(gkf_r2, 4), "mean", round(np.mean(gkf_r2), 4))

## 6. Evaluation: does stuff predict next season better than results do?

In [ ]:
MIN_N = 100
agg = (df.groupby(["season", "pitcher", "pitch_type"])
         .agg(n=("pred", "size"), stuff=("pred", "mean"), actual=("delta_run_exp", lambda s: -s.mean()))
         .mul({"n": 1, "stuff": 100, "actual": 100}).reset_index())  # per 100 pitches
agg = agg[agg.n >= MIN_N]

# Stuff+ scale: 100 = league average for that pitch type, 10 = one SD
g = agg.groupby(["season", "pitch_type"]).stuff
agg["stuff_plus"] = 100 + 10 * (agg.stuff - g.transform("mean")) / g.transform("std")

pairs = agg[agg.season == 2025].merge(agg[agg.season == 2026], on=["pitcher", "pitch_type"], suffixes=("_25", "_26"))
print(f"{len(pairs)} pitcher-pitch types with >= {MIN_N} pitches in both 2025 and 2026")
print("2025 actual RV/100 -> 2026 actual RV/100:", round(pairs.actual_25.corr(pairs.actual_26), 3))
print("2025 stuff  RV/100 -> 2026 actual RV/100:", round(pairs.stuff_25.corr(pairs.actual_26), 3))
print("stuff year-over-year (2025 -> 2026):     ", round(pairs.stuff_25.corr(pairs.stuff_26), 3))

In [ ]:
# 2026 leaderboard (test season, never seen in training or early stopping)
names = raw.drop_duplicates("pitcher").set_index("pitcher").player_name
top = agg[agg.season == 2026].assign(name=lambda d: d.pitcher.map(names))
top.sort_values("stuff_plus", ascending=False).head(15)[["name", "pitch_type", "n", "stuff_plus", "actual"]].round(1)

In [ ]:
import shap
sample = X[te].sample(20_000, random_state=0)
shap_values = shap.TreeExplainer(booster).shap_values(sample)
shap.summary_plot(shap_values, sample, max_display=15)

## 7. Findings → next phase

Fill in after running:
- Which seasons kept (arm_angle coverage):
- Rows dropped per cleaning rule:
- XGBoost vs type-mean on per-pitch RMSE / R²:
- **Stuff → next-year RV correlation vs results → next-year RV:**
- Top SHAP drivers:

Next: per-pitch-type models (fastballs / breaking / offspeed), a location + count Pitching+ variant, and writing Stuff+ back to `pitch_arsenal` (migration 0012).